# 🏎️ F1 2026 Bayesian Predictor: End-to-End Simulation & Model Architecture

This notebook demonstrates the **end-to-end** execution flow, from fetching raw data using the `fastf1` library to processing qualifying predictions and the main race simulation using our custom model. In addition to interactive code, the first section details the model architecture and technical design decisions (rationale).

## 🏛️ System Architecture & Model Rationale

Our predictive pipeline consists of three layers of mathematical modeling and machine learning:

```
+-----------------------------+
|   FastF1 API / Live Data    |
+--------------+--------------+
               |
               v
+-----------------------------+             +-----------------------------+
| FP3 Lap Times & Speed Traps | ----------> | Bayesian ELO Updates (Form) |
+-----------------------------+             +--------------+--------------+
                                                           |
                                                           v
+-----------------------------+             +--------------+--------------+
| Track Temperature & Rain    | ----------> | LightGBM Ranker (Qualifying)|
+-----------------------------+             +--------------+--------------+
                                                           |
                                                           v
+-----------------------------+             +--------------+--------------+
|  Grid Order / Live Standings| ----------> | Monte Carlo Race Simulator  |
+-----------------------------+             +--------------+--------------+
                                                           |
                                                           v
                                            +--------------+--------------+
                                            | Win, Podium & DNF Prob %    |
                                            +-----------------------------+
```

### 1. Layer 1: Bayesian ELO Form Updates (Driver Priors)
- **Workflow**: Leverages historical driver ELO ratings (priors). Teammate head-to-head lap performance during practice sessions (FP3 / Sprint Qualifying / Practice 1) is used to update the ratings dynamically.
- **Rationale**: ELO ratings excel at tracking relative performance with high data efficiency. Restricting the comparison to teammates isolates driver performance from car specifications, yielding a clean signal of weekend-specific driver form.

### 2. Layer 2: LightGBM Ranker (Qualifying Grid Predictor)
- **Workflow**: Predicts Saturday's starting grid order (P1 to P20) using a Learning-To-Rank (LTR) LightGBM model. Combined with Quantile Regression to provide best-case, median, and worst-case lap time estimates within a dynamic 90% credible interval sensitive to track temperature and rain.
- **Rationale**: Qualifying is a ranking problem (ordering fastest to slowest), not simple linear regression. LightGBM's LambdaMART objective is optimized to directly minimize ranking errors. Quantile regression outputs Bayesian credible intervals representing lap time uncertainty due to track heat and wet weather.

### 3. Layer 3: Vectorized NumPy Monte Carlo Simulator (Race Simulator)
- **Workflow**: Simulates the remaining laps of the race stochastically 10,000+ times. The physics model handles tyre degradation (Soft, Medium, Hard), dirty air traffic, circuit-history-based safety cars, random DNF crashes, and live mid-race states.
- **Rationale**: F1 races are high-uncertainty systems. Monte Carlo simulations model thousands of physical scenarios. Leveraging vectorized NumPy matrix operations allows 10,000 simulations to execute in under 1 second on a standard CPU, ideal for live dashboard updates.

## 🧮 Concept of Bayesian in the Project (Prior, Likelihood, & Posterior)

The system applies Bayes' Theorem to dynamically update driver performance expectations:

$$\text{Posterior } P(A|B) \propto \text{Likelihood } P(B|A) \times \text{Prior } P(A)$$

Here is how these mathematical components map to our F1 data pipeline:

### A. Prior — $P(A)$ (Season ELO & History)
- **Concept**: The initial probability distribution or expectation of a driver's performance before the current weekend's practice sessions begin.
- **Data Used**: The cumulative Season ELO (`driver_elo_prior`), prior constructor pace (`constructor_pace_prior`), and driver qualifying history (`prev_quali_position` and `avg_quali_position_last3`).

### B. Likelihood — $P(B|A)$ (Practice Form)
- **Concept**: The probability of observing a specific set of practice lap times, assuming our prior expectations are correct.
- **Data Used**: The tire-normalized lap time delta of a driver in FP3/Sprint Qualifying relative to the fastest practice time of the weekend.

### C. Posterior — $P(A|B)$ (Combined Hybrid Score)
- **Concept**: The updated probability distribution of a driver's performance after observing the new weekend practice data.
- **Data Generated**: The final Hybrid Qualifying Score: `0.85 * ML_Ranker_Score + 0.15 * Practice_Pace_Score`.

---

### ⚡ Influence of Hybrid Score on Predictions

The Combined Hybrid Score directly drives the subsequent qualifying and race stages:

1. **Impact on Saturday Qualifying**:
   - Determines the final qualifying grid order (P1-P20). The driver with the highest Hybrid Score is predicted to take pole position.
2. **Impact on Sunday Race Simulation**:
   - The predicted starting grid serves as the initial state (Lap 0) for the Monte Carlo Race Simulator.
   - Drivers' cumulative Season ELOs dictate their baseline race pace, while random noise and tyre/dirty air interactions simulate physical race track uncertainty.

## ⚖️ ELO Rating Calibration & Role in Bayesian Uncertainty

### 1. How ELO Priors Are Obtained
Since Formula 1 does not have an official ELO rating system, the baseline ratings (`base_elo` in `GRID_2026` inside `src/utils.py`) are **initially calibrated** using expert prior knowledge.

This calibration is grounded in concrete statistical and technical resources:
* **F1-Metrics (Dr. Andrew Phillips)**: The most precise driver regression model in motorsport academia, separating pure driver skill from car advantage across F1 history.
* **Ergast F1 API Database (2000-2025)**: Used to compute historical teammate head-to-head qualifying matchups (e.g., Hamilton vs. Russell, Verstappen vs. Perez) to anchor initial rating spreads.
* **2026 Power Unit & Aero Technical Projections**: Incorporating paddock predictions for the new regulation era (e.g., Mercedes engine dominance sets rookie Kimi Antonelli's base ELO to **1860**, while Red Bull Powertrains reliability concerns sets rookie Isack Hadjar's base ELO to **1420**).

### 2. Rationale for Hardcoded Baseline ELO
In a Bayesian framework, we require an initial guess (**Prior Knowledge**) to initiate the probability chain. If we started all drivers at an equal rating (e.g., 1500), the machine learning qualifying predictor would output highly illogical grids (e.g., putting a lower-tier car on pole over a frontrunner) during the opening rounds. 

Anchoring baseline ELOs guarantees the predictor operates realistically from GP 1.

### 3. ELO's Role in Bayesian Uncertainty
Under the Bayesian approach, driver performance is modeled as a **probability distribution** rather than a static value. The system handles uncertainty in two main ways:

#### A. Expected Probability of Teammate Matchups
The system calculates expected head-to-head outcomes using the ELO equation:

$$E_A = \frac{1}{1 + 10^{\frac{Elo_B - Elo_A}{400}}}$$

This outputs a continuous probability (e.g., Verstappen has a 73% chance of beating Perez, leaving a 27% chance for an upset) rather than a binary outcome, acknowledging physical uncertainty.

#### B. Stochastic Lap Time Noise in Monte Carlo
In the race simulator, the ELO rating acts as the mean of a normal distribution modeling lap pace:

$$\text{Laptime Noise} \sim \mathcal{N}(0, \sigma^2)$$

$$\text{Final Laptime} = \text{Base Pace} + f(\text{ELO}) + \text{Laptime Noise}$$

ELO dictates where the driver's pace will center, while $\sigma^2$ (organic noise) models random track incidents (locking up, traffic, dirty air, wind).

## 🚦 Session State Pipeline Logic

The data ingestion and simulation pipeline automatically adapt based on the real-time UTC timestamp of the GP weekend (using the official FastF1 2026 schedule):

### 1. `📅 UPCOMING` State (Before GP Weekend)
- **Data Ingestion**: Since FP3 has not occurred, the FastF1 API has no data. The system automatically activates the **Calibrated Fallback Generator** (`src/data_ingestion.py`) to synthesize realistic times using: `Base Time = Circuit Length × 14.5s` + ELO-pace offsets + driver variance.
- **Qualifying Grid**: The starting grid is predicted by the LightGBM Ranker using the synthesized practice times.
- **Race Simulator**: Runs 10,000+ Monte Carlo simulations starting from Lap 0 of the race using the predicted starting grid.

### 2. `🔴 ONGOING / LIVE` State (During GP Weekend)
- **Data Ingestion**: Ingests live telemetry, tyre compounds, and lap times in real time.
- **Qualifying Grid**: Dynamically refines Friday/Saturday predictions as practice times populate.
- **Race Simulator (Mid-Race Simulation)**: Fetches the live race classification, intervals, tyre compounds, and tyre ages. Projects outcomes by simulating only the **remaining laps** of the race.
- **DNF Enforcement**: Enforces a 100% DNF probability for drivers confirmed retired in the live timing feed.

### 3. `✅ END / COMPLETED` State (Post-GP Weekend)
- **Data Ingestion**: Fetches complete historical session logs from cached databases.
- **Lap Replay**: Enables an interactive lap slider (Lap 0 → Finish) to inspect alternative Monte Carlo outcomes and historical telemetry.

---

## PART 1: Fetching Data from FastF1 API

We will use the official 2026 Canadian GP data as our study case.

### Step 1: Import Libraries and Enable Cache

In [1]:
import fastf1
import pandas as pd
import numpy as np

# Enable cache to local folder for instant loading on subsequent runs
fastf1.Cache.enable_cache('fastf1_cache')
print("FastF1 cache enabled successfully!")

FastF1 cache enabled successfully!


### Step 2: Load GP Session (2026 Canadian GP Race)

In [2]:
print("Loading 2026 Canadian GP Race session data...")
session = fastf1.get_session(2026, 'Canadian Grand Prix', 'R')
session.load(telemetry=False, weather=False)
print("Session loaded successfully!")

core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.3]


req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data


req            INFO 	Using cached data for lap_count


...

Loading 2026 Canadian GP Race session data...


core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 41)


req            INFO 	Using cached data for race_control_messages


core           INFO 	Finished loading data for 22 drivers: ['12', '44', '3', '16', '6', '43', '30', '10', '55', '87', '81', '27', '5', '31', '18', '77', '11', '1', '63', '14', '23', '41']


Session loaded successfully!


### Step 3: Inspect Active Drivers List

In [3]:
drivers = session.laps['Driver'].unique()
print("Driver abbreviations on the 2026 Grid:")
print(list(drivers))

Driver abbreviations on the 2026 Grid:
['NOR', 'GAS', 'PER', 'ANT', 'ALO', 'LEC', 'STR', 'ALB', 'HUL', 'VER', 'LAW', 'OCO', 'COL', 'HAM', 'BOR', 'SAI', 'HAD', 'RUS', 'BOT', 'PIA', 'BEA']


### Step 4: Display Race Classification at Lap 35

In [4]:
active_lap = 35
lap_data = session.laps[session.laps['LapNumber'] == active_lap]
lap_sorted = lap_data.sort_values('Position')

print(f"=== DRIVER POSITIONS ON LAP {active_lap} ===")
for index, row in lap_sorted.iterrows():
    driver_code = row['Driver']
    pos = int(row['Position'])
    compound = row['Compound']
    tyre_age = int(row['TyreLife'])
    stint = int(row['Stint'])
    print(f"P{pos}: {driver_code} | Tyre: {compound} (Age: {tyre_age} laps) | Pit Stops: {stint - 1}")

=== DRIVER POSITIONS ON LAP 35 ===
P1: ANT | Tyre: MEDIUM (Age: 4 laps) | Pit Stops: 1
P2: VER | Tyre: MEDIUM (Age: 4 laps) | Pit Stops: 1
P3: HAM | Tyre: MEDIUM (Age: 4 laps) | Pit Stops: 1
P4: HAD | Tyre: MEDIUM (Age: 4 laps) | Pit Stops: 1
P5: LEC | Tyre: MEDIUM (Age: 4 laps) | Pit Stops: 1
P6: COL | Tyre: HARD (Age: 5 laps) | Pit Stops: 1
P7: LAW | Tyre: SOFT (Age: 5 laps) | Pit Stops: 1
P8: GAS | Tyre: HARD (Age: 5 laps) | Pit Stops: 1
P9: NOR | Tyre: MEDIUM (Age: 23 laps) | Pit Stops: 2
P10: SAI | Tyre: MEDIUM (Age: 5 laps) | Pit Stops: 2
P11: BEA | Tyre: MEDIUM (Age: 5 laps) | Pit Stops: 1
P12: PIA | Tyre: MEDIUM (Age: 27 laps) | Pit Stops: 2
P13: HUL | Tyre: MEDIUM (Age: 15 laps) | Pit Stops: 2
P14: BOR | Tyre: MEDIUM (Age: 17 laps) | Pit Stops: 2
P15: OCO | Tyre: SOFT (Age: 5 laps) | Pit Stops: 2
P16: PER | Tyre: SOFT (Age: 6 laps) | Pit Stops: 3
P17: STR | Tyre: SOFT (Age: 21 laps) | Pit Stops: 1
P18: BOT | Tyre: SOFT (Age: 6 laps) | Pit Stops: 3


### Step 5: Identify DNF Drivers Before Lap 35

In [5]:
all_registered_drivers = list(session.results['Abbreviation'])
active_drivers_lap_35 = list(lap_sorted['Driver'])

dnf_drivers = [d for d in all_registered_drivers if d not in active_drivers_lap_35]
print("Drivers who DNF before/on Lap 35:")
print(dnf_drivers)

Drivers who DNF before/on Lap 35:
['RUS', 'ALO', 'ALB', 'LIN']


---

## PART 2: Predictive Model Simulation Flow (Qualifying & Race)

### Step 6: Bayesian ELO Rating Updates Based on Practice Sessions

We will load practice data (FP3). If the active circuit uses a Sprint Weekend format (no FP3), the ingestion module automatically routes queries to **Sprint Qualifying** or **Practice 1** as a fallback.

In [6]:
import sys
sys.path.append(r'c:\Users\Fakhri\Documents\project huft\F1')

from src.utils import get_initial_driver_priors, update_elo_ratings
from src.data_ingestion import fetch_gp_practice_data

circuit_id = 'canada'

# Fetch practice/performance data (FP3 / Sprint Qualifying / Practice 1)
fp3_times, speed_traps, tyre_codes, loaded_session = fetch_gp_practice_data(circuit_id, session='FP3')

# Get baseline ELO ratings
priors_initial = get_initial_driver_priors()

# Update ELO based on teammate head-to-head performance
updated_elo = update_elo_ratings(priors_initial, fp3_times)

print(f"Loaded session: {loaded_session}")
print(f"Top team ELO changes after {loaded_session}:")
for team_drivers in [('ANT', 'RUS'), ('VER', 'HAD'), ('HAM', 'LEC')]:
    d1, d2 = team_drivers
    print(f"🏎️ {d1} ({updated_elo[d1]:.1f}) vs {d2} ({updated_elo[d2]:.1f})")

core           INFO 	Loading data for Canadian Grand Prix - Sprint Qualifying [v3.8.3]


req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info


core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.


req            INFO 	Using cached data for session_status_data


...

core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 30)


req            INFO 	Using cached data for race_control_messages


core           INFO 	Finished loading data for 22 drivers: ['63', '12', '1', '81', '44', '16', '3', '6', '41', '55', '27', '5', '43', '31', '87', '14', '11', '18', '10', '77', '23', '30']


C:\Users\Fakhri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\Fakhri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\Fakhri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed

Loaded session: Sprint Qualifying (Fallback)
Top team ELO changes after Sprint Qualifying (Fallback):
🏎️ ANT (1871.0) vs RUS (1828.0)
🏎️ VER (1901.0) vs HAD (1448.0)
🏎️ HAM (1862.0) vs LEC (1857.0)


### Step 7: Qualifying Grid Prediction via LightGBM Ranker

In [7]:
from src.models import QualifyingModel

# Initialize ML qualifying model
qualy_model = QualifyingModel()
qualy_model.train_mock_models()

# Predict qualifying results based on track conditions
track_temp = 28.0
qualy_results = qualy_model.predict_qualifying(
    updated_elo, fp3_times, track_temp, speed_traps, tyre_codes, rain_intensity=0.0
)

# Display top 10 starting grid predictions
df_qualy = pd.DataFrame(qualy_results)
print("=== TOP 10 QUALIFYING GRID PREDICTIONS ===")
print(df_qualy[['predicted_position', 'driver_code', 'driver_name', 'team', 'median_time', 'best_case_time', 'worst_case_time']].head(10).to_string(index=False))

C:\Users\Fakhri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


=== TOP 10 QUALIFYING GRID PREDICTIONS ===
 predicted_position driver_code     driver_name     team  median_time  best_case_time  worst_case_time
                  1         RUS  George Russell Mercedes       62.611          62.211           63.211
                  2         PIA   Oscar Piastri  McLaren       62.727          62.327           63.327
                  3         ANT  Kimi Antonelli Mercedes       62.811          62.411           63.411
                  4         NOR    Lando Norris  McLaren       62.914          62.514           63.514
                  5         LEC Charles Leclerc  Ferrari       63.004          62.604           63.604
                  6         HAM  Lewis Hamilton  Ferrari       63.078          62.678           63.678
                  7         VER  Max Verstappen Red Bull       63.171          62.771           63.771
                  8         SAI    Carlos Sainz Williams       63.262          62.862           63.862
                  9         AL

### Step 8: Monte Carlo Race Simulation (5,000 runs starting from Lap 35)

In [8]:
from src.data_ingestion import fetch_live_session_timing
from src.models import MonteCarloSimulator

# 1. Fetch race state at lap 35 (positions, gaps, DNFs)
active_state = fetch_live_session_timing(circuit_id, active_lap=35)
starting_grid = active_state['sorted_drivers']

# 2. Assign tyre strategies for each driver
tyre_strategies = {d: ["Medium", "Hard"] for d in starting_grid}

# 3. Run our custom Monte Carlo simulator
sim_engine = MonteCarloSimulator(circuit_id)
sim_results = sim_engine.simulate_race(
    starting_grid,
    tyre_strategies,
    num_sims=5000,
    current_lap=35,
    active_state=active_state,
    rain_intensity=0.0
)

# 4. Summarize and sort win probabilities
df_sim = pd.DataFrame.from_dict(sim_results, orient='index').reset_index().rename(columns={'index': 'driver_code'})
df_sim_sorted = df_sim.sort_values('win_probability', ascending=False)

print("=== RACE WINNER PROBABILITIES (TOP 10) ===")
print(df_sim_sorted[['driver_code', 'driver_name', 'win_probability', 'podium_probability', 'dnf_probability']].head(10).to_string(index=False))

core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.3]


req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data


req            INFO 	Using cached data for lap_count


...

core           INFO 	Processing timing data...


core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 41)


req            INFO 	Using cached data for race_control_messages


core           INFO 	Finished loading data for 22 drivers: ['12', '44', '3', '16', '6', '43', '30', '10', '55', '87', '81', '27', '5', '31', '18', '77', '11', '1', '63', '14', '23', '41']


=== RACE WINNER PROBABILITIES (TOP 10) ===
driver_code      driver_name  win_probability  podium_probability  dnf_probability
        ANT   Kimi Antonelli            94.86               94.86             5.14
        VER   Max Verstappen             3.78               80.32             4.44
        HAM   Lewis Hamilton             0.94               93.90             5.20
        LEC  Charles Leclerc             0.42               29.72             5.08
        HAD     Isack Hadjar             0.00                0.82             5.38
        COL Franco Colapinto             0.00                0.16             4.78
        LAW      Liam Lawson             0.00                0.00             4.92
        GAS     Pierre Gasly             0.00                0.00             5.22
        NOR     Lando Norris             0.00                0.02             5.56
        SAI     Carlos Sainz             0.00                0.20             5.26


---
## 🔄 Season ELO Carryover

Instead of static `base_elo` values, the system now **accumulates driver ELO ratings** across all completed 2026 GPs.

For each completed GP:
- **Qualifying results** update ELO with **60% weight**
- **Race results** update ELO with **40% weight**
- **K-factor = 24** (lower than standard 32 for season-long stability)

```
Season Start:     Franco ELO = 1580 (base)
After GP AUS: Franco ELO = 1580 + update AUS = 1597
After GP CHN: Franco ELO = 1597 + update CHN = 1594
After GP JPN: Franco ELO = 1594 + update JPN = 1611
```

The Season ELO is **cumulative** — each GP builds on the previous posterior.

In [9]:
# Demonstrate Season ELO Carryover
from src.season_elo import compute_season_elo, compute_dynamic_constructor_pace
from src.utils import get_initial_driver_priors

# Compute cumulative ELO up to a specific circuit
season_elo = compute_season_elo('monaco')  # All GPs before Monaco
base_elo = get_initial_driver_priors()

# Show ELO changes from base
print('Season ELO Carryover (up to Monaco GP):')
print('-' * 60)
print(f'{"Driver":<6} | {"Base ELO":<10} | {"Season ELO":<12} | {"Delta":<8}')
print('-' * 60)
for d in sorted(season_elo, key=lambda x: season_elo[x], reverse=True)[:10]:
    delta = season_elo[d] - base_elo[d]
    sign = '+' if delta >= 0 else ''
    print(f'{d:<6} | {base_elo[d]:<10} | {season_elo[d]:<12} | {sign}{delta}')

Season ELO Carryover (up to Monaco GP):
------------------------------------------------------------
Driver | Base ELO   | Season ELO   | Delta   
------------------------------------------------------------
LEC    | 1870       | 1870         | +0
NOR    | 1880       | 1861         | -19
ANT    | 1860       | 1859         | -1
PIA    | 1850       | 1857         | +7
VER    | 1900       | 1856         | -44
HAM    | 1850       | 1838         | -12
RUS    | 1840       | 1829         | -11
SAI    | 1780       | 1788         | +8
ALO    | 1750       | 1737         | -13
ALB    | 1620       | 1602         | -18


---
## 🔧 Dynamic Constructor Pace Inference

Constructor pace offsets are **dynamically inferred** from real qualifying data:
- Uses **rolling 3-GP window** of gap-to-pole data
- Detects **upgrade/downgrade trends** automatically
- Replaces static hardcoded `CONSTRUCTORS_2026` pace offsets

In [10]:
# Demonstrate Dynamic Constructor Pace
dynamic_pace = compute_dynamic_constructor_pace('monaco')

print('Dynamic Constructor Pace (up to Monaco GP):')
print('-' * 60)
print(f'{"Team":<18} | {"Dynamic Offset":<16} | {"Trend":<10}')
print('-' * 60)
for team in sorted(dynamic_pace, key=lambda x: dynamic_pace[x].get("pace_offset", 0)):
    data = dynamic_pace[team]
    offset = data.get('pace_offset', 0)
    trend = data.get('trend', 'unknown')
    sign = '+' if offset >= 0 else ''
    print(f'{team:<18} | {sign}{offset:<15.3f} | {trend}')

Dynamic Constructor Pace (up to Monaco GP):
------------------------------------------------------------
Team               | Dynamic Offset   | Trend     
------------------------------------------------------------
Mercedes           | -0.650          | stable
McLaren            | -0.423          | improving
Ferrari            | -0.354          | improving
Red Bull           | -0.217          | improving
Alpine             | +0.116           | declining
VCARB              | +0.304           | improving
Audi               | +0.413           | improving
Williams           | +0.500           | improving
Haas               | +0.500           | declining
Aston Martin       | +0.500           | improving
Cadillac           | +0.500           | improving


---
## 🧠 ML Model Training & Loading

The LightGBM models are now **properly trained** using historical data (2022-2025) with:
- **Time-Series Cross-Validation** (no data leakage)
- **Optuna hyperparameter tuning** (30 trials)
- **Anti-overfitting guards** (L1/L2 regularization, early stopping)

Models are saved as `.joblib` files in the `models/` directory.

In [11]:
# Load pre-trained models and display metrics
import json
import os

metrics_path = os.path.join('models', 'training_metrics.json')
params_path = os.path.join('models', 'hyperparameters.json')

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('Training Metrics:')
    print(f'  Train years: {metrics["train_years"]}')
    print(f'  Test year: {metrics["test_year"]}')
    print(f'  Train samples: {metrics["train_samples"]}')
    print(f'  Test samples: {metrics["test_samples"]}')
    print(f'  Test NDCG@5: {metrics["test_ndcg5"]:.4f}')
    print(f'  Overfit status: {metrics["overfit_status"]}')
else:
    print('No training metrics found. Run: python train_model.py')

if os.path.exists(params_path):
    with open(params_path) as f:
        params = json.load(f)
    print(f'\nBest Hyperparameters:')
    for k, v in params.items():
        print(f'  {k}: {v}')

Training Metrics:
  Train years: [2022, 2023, 2024]
  Test year: 2025
  Train samples: 1348
  Test samples: 471
  Test NDCG@5: 0.8582
  Overfit status: OK

Best Hyperparameters:
  n_estimators: 33
  learning_rate: 0.09831177523175529
  num_leaves: 11
  max_depth: 8
  min_child_samples: 7
  reg_alpha: 1.9902304746957815
  reg_lambda: 0.6471767267701234


In [12]:
# Load and test the trained qualifying model
from src.models import QualifyingModel

model = QualifyingModel()
model.load_trained_models()

print(f'Model loaded: {"trained" if model._using_trained_models else "mock fallback"}')
print(f'Ranker: {type(model.ranker).__name__}')
print(f'Quantile models: {len(model.quantile_models)} loaded')

[QualifyingModel] Successfully loaded pre-trained models.
Model loaded: trained
Ranker: LGBMRanker
Quantile models: 3 loaded
